# 09 · Endoscapes-Seg 로 cystic_duct / cystic_artery 학습 (처음부터)

> **공부 기록 노트북 9번.** 노트북 08에서 **CholecSeg8k 가 cystic_duct 를 사실상
> 라벨하지 않음**(train 전체 5840 프레임에 3 px)을 확인했습니다. 그래서 CVS 핵심
> 두 구조(duct + artery)는 **Endoscapes2023 segmentation** 에서 학습합니다
> (duct 2.9M px, artery 1.1M px — 진짜 라벨).

이 노트북은 **빈 Colab 에서 끝까지** 돌아갑니다:

1. 코드 받기 + 설치
2. Endoscapes 다운로드 (~5.9 GB)
3. **매핑 검증** — duct/artery 가 기대한 id 에 있는지 (CholecSeg8k 색상맵 실수 방지)
4. 스모크 (연결 1~2분)
5. **본학습** — duct/artery 에 진짜 라벨이 있는 상태에서 Focal-Tversky·샘플러·copy-paste 적용
6. 결과 — `cystic_duct/artery Dice` 와 `[입력 | 정답 | 예측]` 그림

⚠️ **Runtime → Change runtime type → GPU(T4)** 먼저! (SAM2 는 GPU 필수)

## 0. 코드 받기 + 설치

빈 Colab 이므로 코드를 받고 라이브러리를 깝니다. 이 수정들이 들어있는 **기능
브랜치**를 받습니다 (머지 후엔 `BRANCH="main"`).

In [ ]:
%cd /content
import os
BRANCH = "claude/stoic-brown-jllvfb"
if not os.path.isdir("surgical-ai"):
    !git clone -b $BRANCH https://github.com/duck-bin/surgical-ai.git
%cd surgical-ai
!git fetch origin $BRANCH && git checkout $BRANCH && git pull --ff-only origin $BRANCH
!bash scripts/colab_setup.sh

import torch
print("준비 완료 ·", os.getcwd())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "❌ 없음 → Runtime > Change runtime type > GPU 로 바꾸고 이 셀 다시 실행")

## 1. Endoscapes2023 다운로드 (~5.9 GB)

CAMMA 공개 미러에서 받습니다. `-c` 라 중간에 끊겨도 이어받습니다. 압축은
`./data/endoscapes2023/endoscapes/` 로 풀립니다 (`train_seg/`, `semseg/` 등).

⏱️ 다운로드 5~10분 + 압축 해제 몇 분. 이미 받았으면 빠르게 넘어갑니다.

In [ ]:
import os
# 레포 보장 + 작업 디렉터리 고정 (셀 0 을 건너뛰었거나 커널이 리셋돼도 안전)
os.chdir("/content")
if not os.path.isdir("/content/surgical-ai"):
    !git clone -b claude/stoic-brown-jllvfb https://github.com/duck-bin/surgical-ai.git
os.chdir("/content/surgical-ai")

DEST = "data/endoscapes2023"
os.makedirs(DEST, exist_ok=True)
if os.path.isfile("scripts/download_endoscapes.sh"):
    !bash scripts/download_endoscapes.sh
else:   # 스크립트가 없어도 직접 받기 (스크립트 내용과 동일)
    !wget -c -O {DEST}/endoscapes.zip https://s3.unistra.fr/camma_public/datasets/endoscapes/endoscapes.zip
    !unzip -q -o {DEST}/endoscapes.zip -d {DEST}

ok = os.path.isdir(f"{DEST}/endoscapes/semseg")
print("semseg 폴더 존재:", ok)
assert ok, "압축 해제 실패 — 위 로그 확인"

## 2. 매핑 검증 (학습 전 필수)

CholecSeg8k 색상맵 실수를 되풀이하지 않으려면, **디스크의 실제 semseg id** 가
우리가 가정한 매핑과 맞는지 먼저 확인합니다. duct(4)·artery(3) 에 의미있는
픽셀수가 보여야 합니다.

In [ ]:
from src.data.endoscapes_seg import semseg_id_histogram, ENDOSCAPES_SEMSEG_ID_TO_NAME

hist = semseg_id_histogram("./data/endoscapes2023/endoscapes/semseg", sample=300)
print("semseg id → 픽셀수:")
for i, n in hist.items():
    print(f"  {i:>3}: {n:>12}   {ENDOSCAPES_SEMSEG_ID_TO_NAME.get(i, '*** 미등록(→배경) ***')}")
print(f"\ncystic_duct(4): {hist.get(4,0):,} px   cystic_artery(3): {hist.get(3,0):,} px")
assert hist.get(4,0) > 0 and hist.get(3,0) > 0, "duct/artery 가 안 보임 — 매핑/다운로드 확인"
print("✅ 매핑 OK — duct/artery 라벨 존재 확인")

## 3. 모델 선택 + 스모크 (연결 1~2분)

`MODEL` 을 고릅니다:
- **`unet_baseline`** — 가볍고 빠름. *"duct/artery 가 학습되는가"* 를 몇십 분 안에
  확인하기 좋음 (T4 권장 출발점).
- **`sam2_lora`** — 실제 연구 모델. 더 정확하지만 느림 (A100 권장).

스모크는 20 배치만 돌려 **에러 없이 끝까지 연결되는지**만 봅니다 (점수 무의미).

In [ ]:
MODEL = "unet_baseline"   # 빠른 검증. 실제 모델은 "sam2_lora" (느림, A100 권장)

!python -u -m src.train.train_segmentation \
    data=endoscapes2023_seg model={MODEL} \
    epochs=1 limit_batches=20 data.image_size=256 num_workers=2 \
    low_memory=true batch_size=2 \
    sampler=null loss.class_weighting=none copy_paste.enabled=false

## 4. 본학습

이제 진짜 학습입니다. 이번엔 duct/artery 에 **진짜 라벨**이 있으니, 제가 넣은
세 장치가 비로소 효과를 냅니다:

- **Focal-Tversky** loss (β>α) — 작은 구조의 *놓침* 을 더 세게 벌함
- **가중 샘플러** — duct/artery 가 든 프레임을 더 자주 뽑음
- **copy-paste** (`copy_paste.enabled=true`) — duct/artery 패치를 다른 프레임에 합성

학습 시작 직후 `[copy-paste] harvested N ...` (패치 수집), 끝에
**`test_cystic_duct_dice` / `test_cystic_artery_dice`** 가 출력됩니다 — 그게 핵심.

⏱️ U-Net @512 는 T4 에서 수십 분~1시간대(early-stopping). SAM2 는 수 시간(A100 권장).
중단돼도 같은 셀 재실행하면 `outputs/{MODEL}/last.ckpt` 에서 이어집니다.
> OOM 나면 `data.image_size=384` 로, 너무 느리면 `data.image_size=384` 또는 A100 사용.

In [ ]:
!python -u -m src.train.train_segmentation \
    data=endoscapes2023_seg model={MODEL} \
    low_memory=true data.image_size=512 num_workers=2 \
    copy_paste.enabled=true wandb.mode=disabled

## 5. 결과 — duct/artery 가 잡혔는가?

학습 로그 맨 끝의 `test_cystic_duct_dice` / `test_cystic_artery_dice` 가 1차
답입니다. 아래는 val 프레임에서 **눈으로** 확인합니다: duct/artery 가 든 프레임을
골라 `[입력 | 정답 | 예측]` 으로 그립니다. 예측의 duct(노랑)/artery(연두)가 정답
위치를 따라가면 성공입니다.

In [ ]:
import torch, numpy as np
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
from src.train.train_segmentation import build_model
from src.eval.benchmark_runner import load_segmentation_checkpoint
from src.eval.metrics import dice_score
from src.data.endoscapes_seg import EndoscapesSegDataset
from src.data.transforms import build_eval_transforms
from src.data.cholecseg8k import CLASS_NAMES, NUM_CLASSES
from src.utils.viz import overlay_mask

IMG = 512
DUCT, ARTERY = CLASS_NAMES.index("cystic_duct"), CLASS_NAMES.index("cystic_artery")
device = "cuda" if torch.cuda.is_available() else "cpu"

model = build_model(OmegaConf.load(f"configs/model/{MODEL}.yaml"), pretrained=False)
model = load_segmentation_checkpoint(model, f"outputs/{MODEL}/best.ckpt").to(device).eval()

ds = EndoscapesSegDataset(split="val", image_size=IMG, transform=build_eval_transforms(IMG))

# duct 또는 artery 가 든 val 프레임 몇 장
idx = [i for i in range(len(ds))
       if np.isin([DUCT, ARTERY], ds[i]["mask"].numpy()).any()][:3]
print(f"val {len(ds)} 프레임 중 duct/artery 포함 예시 {len(idx)}장")

fig, ax = plt.subplots(len(idx), 3, figsize=(13, 4*len(idx))); ax = np.atleast_2d(ax)
for r, i in enumerate(idx):
    s = ds[i]; img = s["image"].unsqueeze(0).to(device)
    with torch.no_grad():
        pred = model(img).argmax(1)[0].cpu().numpy()
    rgb = s["image"].permute(1,2,0).numpy(); rgb = ((rgb-rgb.min())/(rgb.max()-rgb.min()+1e-8)*255).astype("uint8")
    gt = s["mask"].numpy()
    ax[r,0].imshow(rgb); ax[r,0].set_title("input")
    ax[r,1].imshow(overlay_mask(rgb, gt)); ax[r,1].set_title("ground truth")
    ax[r,2].imshow(overlay_mask(rgb, pred)); ax[r,2].set_title(f"{MODEL} pred")
    for a in ax[r]: a.axis("off")
plt.tight_layout(); plt.show()
print("색: duct=노랑, artery=연두, gallbladder=하늘, tool=회색")

# val 일부에서 per-class Dice (NaN 무시 평균)
per = {c: [] for c in range(NUM_CLASSES)}
for i in range(0, len(ds), max(1, len(ds)//80)):
    s = ds[i]; img = s["image"].unsqueeze(0).to(device)
    with torch.no_grad():
        pred = model(img).argmax(1)[0].cpu()
    d, _ = dice_score(pred, s["mask"], NUM_CLASSES)
    for c in range(NUM_CLASSES):
        if not np.isnan(d[c]): per[c].append(d[c])
print("\nval per-class Dice (표본):")
for c, name in enumerate(CLASS_NAMES):
    vals = per[c]
    print(f"  {name:14s}: {np.mean(vals):.3f}" if vals else f"  {name:14s}: (해당 클래스 없음)")

## 마무리 — 해석

- **`cystic_duct` / `cystic_artery` Dice 가 0 을 벗어나면** → 피벗 성공. CholecSeg8k
  로는 불가능했던 학습이 Endoscapes 의 진짜 라벨 + (Focal-Tversky·샘플러·copy-paste)
  로 가능해진 것입니다. 다음은 **SAM2(`MODEL="sam2_lora"`)로 품질 끌어올리기** 또는
  **CVS 평가(`train_cvs` → benchmark)** 로.
- **여전히 낮으면** → `copy_paste.paste_prob`↑, `loss.tversky_beta`↑(0.7→0.8),
  `optimizer.lr_decoder`, `epochs` 등을 조정. 결과를 공유해 주세요.

> 참고: Endoscapes 엔 **liver 라벨이 없어** liver 채널은 비어 있습니다(예상된 것).
> CVS 기준엔 liver 가 안 들어가 문제 없지만, 6클래스 전체가 필요하면 나중에
> CholecSeg8k 와 partial-label joint 학습으로 보강합니다.